# ATP 2-01.3 — Fine-Tuned LLM (Google Colab / T4 GPU)

Fine-tune a language model on ATP 2-01.3 (Intelligence Preparation of the Battlefield)
to serve as a doctrine assistant for military intelligence questions.

**Hardware:** Google Colab free tier (T4 GPU)  
**Model:** Gemma 2-2B-IT (unsloth/gemma-2-2b-it)  
**Framework:** Unsloth + QLoRA  

```
PDF → Chunks → QA Pairs → Format → Train → Evaluate → Export GGUF
  1       2        3         4        5         6           7
```

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Accept the [Gemma 2 license](https://huggingface.co/google/gemma-2-2b-it) on HuggingFace
3. Have a HuggingFace **read token** ready (huggingface.co/settings/tokens)
4. Have `ATP_2-01.3.pdf` ready to upload

## Step 0 — GPU Check

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU found — Runtime → Change runtime type → T4 GPU"
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"PyTorch : {torch.__version__}")

## Step 1 — Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl==0.8.6" peft accelerate bitsandbytes
!pip install pdfplumber rouge-score matplotlib datasets
!pip install --upgrade "huggingface_hub[cli]"

## Step 2 — Mount Drive & Upload PDF

In [ ]:
from google.colab import drive, files
import os

drive.mount('/content/drive')

# ── Path constants ────────────────────────────────────────────
DRIVE_BASE   = '/content/drive/MyDrive/atp-finetuning'
PDF_PATH     = '/content/ATP_2-01.3.pdf'
CHUNKS_PATH  = f'{DRIVE_BASE}/data/chunks.jsonl'
SEEDS_PATH   = f'{DRIVE_BASE}/data/seeds.jsonl'
TRAIN_PATH   = f'{DRIVE_BASE}/data/train.jsonl'
VALID_PATH   = f'{DRIVE_BASE}/data/val.jsonl'
ADAPTER_PATH = f'{DRIVE_BASE}/outputs/atp_gemma2_adapter'
EVAL_PATH    = f'{DRIVE_BASE}/eval/results/atp_gemma2_colab.json'
CHART_PATH   = f'{DRIVE_BASE}/eval/score_chart.png'
GGUF_DIR     = f'{DRIVE_BASE}/burns'
MODEL_NAME   = 'unsloth/gemma-2-2b-it'

for d in [f'{DRIVE_BASE}/data', f'{DRIVE_BASE}/outputs',
          f'{DRIVE_BASE}/eval/results', GGUF_DIR]:
    os.makedirs(d, exist_ok=True)

# Upload PDF
print("Upload ATP_2-01.3.pdf when prompted:")
uploaded = files.upload()
for fname, data in uploaded.items():
    with open(PDF_PATH, 'wb') as f:
        f.write(data)
    print(f"Saved → {PDF_PATH} ({len(data):,} bytes)")

## Step 3 — HuggingFace Login

In [ ]:
from huggingface_hub import login
login()  # Paste your HF read token when prompted

## Step 4 — Chunk PDF

Parses ATP 2-01.3 into paragraph-level chunks with chapter/appendix metadata.
Output saved to Drive so this step can be skipped on resume.

In [ ]:
import json, re, os
from pathlib import Path
import pdfplumber

ATP_CHAPTERS = {
    1: 'IPB Fundamentals',
    2: 'IPB Support to Planning',
    3: 'Step 1 - Define the Operational Environment',
    4: 'Step 2 - Describe Environmental Effects',
    5: 'Step 3 - Evaluate the Threat',
    6: 'Step 4 - Determine Threat COAs',
    7: 'Unified Action and Unique Environments',
    8: 'Multi-Domain Considerations',
}

PARA_ID_RE   = re.compile(r'^(\d+)-(\d+)\.\s+', re.MULTILINE)
APPEND_ID_RE = re.compile(r'^([A-D])-(\d+)\.\s+', re.MULTILINE)

def extract_text(pdf_path):
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ''
            pages.append(text)
    return '\n'.join(pages)

def detect_chapter(para_id):
    m = PARA_ID_RE.match(para_id + '. ')
    if m:
        ch = int(m.group(1))
        return ch, ATP_CHAPTERS.get(ch, f'Chapter {ch}')
    return None, 'Appendix'

def chunk_text(text):
    chunks = []
    chunk_id = 0
    # Split on paragraph IDs
    parts = re.split(r'(?=^(?:\d+-\d+|[A-D]-\d+)\.\s)', text, flags=re.MULTILINE)
    for part in parts:
        part = part.strip()
        if not part or len(part.split()) < 10:
            continue
        # Extract para_id from start of chunk
        m = re.match(r'^(\d+-\d+|[A-D]-\d+)\.\s+', part)
        para_id = m.group(1) if m else f'unk-{chunk_id}'
        ch_num, ch_title = detect_chapter(para_id)
        chunks.append({
            'chunk_id':    f'chunk_{chunk_id:04d}',
            'para_id':     para_id,
            'text':        part,
            'chapter_num': ch_num,
            'chapter_title': ch_title,
            'word_count':  len(part.split()),
        })
        chunk_id += 1
    return chunks

if os.path.exists(CHUNKS_PATH):
    print(f'Chunks already exist at {CHUNKS_PATH} — skipping. Delete to re-chunk.')
else:
    print('Extracting text from PDF...')
    text   = extract_text(PDF_PATH)
    chunks = chunk_text(text)
    with open(CHUNKS_PATH, 'w') as f:
        for c in chunks:
            f.write(json.dumps(c) + '\n')
    print(f'Saved {len(chunks)} chunks → {CHUNKS_PATH}')

chunks = [json.loads(l) for l in open(CHUNKS_PATH)]
print(f'Total chunks: {len(chunks)}')
print('\nSample:')
print(json.dumps(chunks[0], indent=2)[:400])

## Step 5 — Generate QA Pairs

Uses the model itself to generate QA pairs from each chunk.
Appends to Drive so the run is safely resumable after disconnects.

> **Note:** Targeting 300 pairs for Colab (vs 5,000 for the full MLX run). Adjust `TARGET` as needed.

In [ ]:
import json, random, os, gc, re
import torch
from unsloth import FastLanguageModel

TARGET     = 50
BATCH_SIZE = 5

QUESTION_TYPES = [
    'factual', 'procedural', 'conceptual', 'application',
    'comparative', 'scenario', 'definition', 'relationship',
]

GEN_SYSTEM = (
    "You are a military doctrine expert generating training data for ATP 2-01.3 "
    "(Intelligence Preparation of the Battlefield). Generate a QA pair from the "
    "given doctrine text. The answer must end with [Reference: ATP 2-01.3, para X-Y]."
)

def is_quality_chunk(text):
    words = text.split()
    if len(words) < 30:
        return False
    # Require ≥70% of chars to be letters, spaces, digits, or common punctuation
    good = sum(c.isalpha() or c.isspace() or c.isdigit() or c in '.,;:!?()\'-' for c in text)
    return (good / max(len(text), 1)) >= 0.70

def make_gen_prompt(chunk_text, q_type):
    return (
        f"<start_of_turn>user\n"
        f"{GEN_SYSTEM}\n\n"
        f"Doctrine text:\n{chunk_text[:600]}\n\n"
        f"Generate a {q_type} question and answer about this doctrine. "
        f"Format:\nQUESTION: <question>\nANSWER: <answer>\n"
        f"<end_of_turn>\n<start_of_turn>model\n"
    )

def parse_qa(text, chunk, q_type, qa_id):
    cleaned = re.sub(r'\*+', '', text)
    q_match = re.search(r'QUESTION\s*:\s*(.+?)(?=ANSWER\s*:|$)', cleaned, re.DOTALL | re.IGNORECASE)
    a_match = re.search(r'ANSWER\s*:\s*(.+)',                     cleaned, re.DOTALL | re.IGNORECASE)
    if not q_match or not a_match:
        return None
    q = q_match.group(1).strip().split('\n')[0].strip()
    a = a_match.group(1).strip()
    if len(q.split()) < 5 or len(a.split()) < 10:
        return None
    return {
        'qa_id':         qa_id,
        'source_chunks': [chunk['para_id']],
        'question_type': q_type,
        'question':      q,
        'answer':        a,
        'metadata': {
            'chapter_num':   chunk.get('chapter_num'),
            'chapter_title': chunk.get('chapter_title'),
        }
    }

# Count existing seeds
existing = 0
if os.path.exists(SEEDS_PATH):
    existing = sum(1 for _ in open(SEEDS_PATH))
print(f'Existing seeds: {existing} / {TARGET}')

if existing >= TARGET:
    print('Target reached — skipping generation.')
else:
    print('Loading model for generation...')
    gen_model, gen_tok = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True,
    )
    FastLanguageModel.for_inference(gen_model)

    all_chunks    = [json.loads(l) for l in open(CHUNKS_PATH)]
    good_chunks   = [c for c in all_chunks if is_quality_chunk(c['text'])]
    print(f'Quality chunks: {len(good_chunks)} / {len(all_chunks)} (filtered {len(all_chunks)-len(good_chunks)} bad)')
    random.shuffle(good_chunks)

    qa_count   = existing
    fail_count = 0

    with open(SEEDS_PATH, 'a') as out_f:
        for chunk in good_chunks:
            if qa_count >= TARGET:
                break
            q_type = random.choice(QUESTION_TYPES)
            prompt = make_gen_prompt(chunk['text'], q_type)
            inputs = gen_tok(prompt, return_tensors='pt').to('cuda')
            with torch.no_grad():
                out = gen_model.generate(
                    **inputs, max_new_tokens=300, temperature=0.7,
                    do_sample=True, pad_token_id=gen_tok.eos_token_id,
                )
            decoded = gen_tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
            seed = parse_qa(decoded, chunk, q_type, f'qa_{qa_count:04d}')
            if seed:
                out_f.write(json.dumps(seed) + '\n')
                qa_count += 1
                if qa_count % 10 == 0:
                    print(f'  Generated {qa_count}/{TARGET}...')
            else:
                fail_count += 1
                if fail_count <= 2:
                    print(f'  [parse fail #{fail_count}] sample: {repr(decoded[:150])}')

    del gen_model, gen_tok
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Done — {qa_count} seeds saved ({fail_count} parse failures)')

seeds = [json.loads(l) for l in open(SEEDS_PATH)]
from collections import Counter
qtypes = Counter(s.get('question_type') for s in seeds)
print(f'\nTotal QA pairs: {len(seeds)}')
for qt, cnt in sorted(qtypes.items(), key=lambda x: -x[1]):
    print(f'  {qt:<20}: {cnt}')


## Step 6 — Format Training Data

Converts QA pairs into Gemma 2 chat-template format. Splits 90/10 train/val.

In [ ]:
import json, random

VAL_RATIO = 0.10
random.seed(42)

SYSTEM_PROMPT = (
    "You are a doctrine-grounded military intelligence assistant specializing in "
    "ATP 2-01.3 (Intelligence Preparation of the Battlefield, March 2019). "
    "Provide accurate, doctrinally grounded answers. "
    "Place citations at the END in [Reference: ATP 2-01.3, para X-Y] format."
)

def format_example(seed):
    q   = seed.get('question', '').strip()
    ans = seed.get('answer',   '').strip()
    if not q or not ans:
        return None
    text = (
        f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{q}<end_of_turn>\n"
        f"<start_of_turn>model\n{ans}<end_of_turn>"
    )
    return {'text': text, 'question_type': seed.get('question_type'),
            'chapter': seed.get('metadata', {}).get('chapter_title')}

seeds = [json.loads(l) for l in open(SEEDS_PATH)]
examples = [e for s in seeds if (e := format_example(s)) is not None]
random.shuffle(examples)

n_val   = max(1, int(len(examples) * VAL_RATIO))
n_train = len(examples) - n_val
train_ex = examples[:n_train]
val_ex   = examples[n_train:]

with open(TRAIN_PATH, 'w') as f:
    for ex in train_ex: f.write(json.dumps(ex) + '\n')
with open(VALID_PATH, 'w') as f:
    for ex in val_ex:   f.write(json.dumps(ex) + '\n')

print(f'Train: {n_train} | Val: {n_val}')
print('\nFormatted example (first 500 chars):')
print(train_ex[0]['text'][:500])

## Step 7 — Train (Unsloth QLoRA)

Fine-tunes Gemma 2-2B with QLoRA. Checkpoints saved to Drive.

Expected time on T4: ~40-60 minutes for 300 examples / 2 epochs.

In [ ]:
import json, os
import torch
from datasets import Dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

MAX_SEQ_LENGTH = 2048
CKPT_DIR       = f'{DRIVE_BASE}/outputs/checkpoints'

# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH,
    dtype=None, load_in_4bit=True,
)

# Apply QLoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.0, bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42, use_rslora=True,
)
model.print_trainable_parameters()

# Load datasets
def load_jsonl(path):
    rows = []
    with open(path) as f:
        for line in f:
            obj = json.loads(line)
            if 'text' in obj:
                rows.append({'text': obj['text']})
    return Dataset.from_list(rows)

train_ds = load_jsonl(TRAIN_PATH)
val_ds   = load_jsonl(VALID_PATH)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds,
    args=SFTConfig(
        output_dir=CKPT_DIR,
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.05,
        optim='adamw_8bit',
        weight_decay=0.01,
        bf16=False, fp16=True,
        logging_steps=10,
        eval_strategy='steps', eval_steps=50,
        save_strategy='epoch',
        load_best_model_at_end=False,
        seed=42, report_to='none',
        packing=True, dataset_num_proc=2,
    ),
)

stats = trainer.train()
print(f"\nTrain loss: {stats.metrics.get('train_loss', 'N/A'):.4f}")

# Save adapter to Drive
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print(f'Adapter saved → {ADAPTER_PATH}')

## Step 8 — Evaluate (ROUGE-L)

Runs ROUGE-L evaluation against the validation split — base model vs fine-tuned.
Same approach as the MLX pipeline for a fair comparison.

In [ ]:
import gc, json, os, re
import torch
import numpy as np
from unsloth import FastLanguageModel
from rouge_score import rouge_scorer as rs_module

# Path fallback for runtime restarts
if 'DRIVE_BASE' not in dir():
    DRIVE_BASE   = '/content/drive/MyDrive/atp-finetuning'
    VALID_PATH   = f'{DRIVE_BASE}/data/val.jsonl'
    ADAPTER_PATH = f'{DRIVE_BASE}/outputs/atp_gemma2_adapter'
    EVAL_PATH    = f'{DRIVE_BASE}/eval/results.jsonl'
    MODEL_NAME   = 'unsloth/gemma-2-2b-it'

EVAL_SYSTEM = (
    "You are a doctrine-grounded military intelligence assistant specializing in "
    "ATP 2-01.3 (Intelligence Preparation of the Battlefield, March 2019). "
    "Provide accurate, doctrinally grounded answers. "
    "Place citations at the END in [Reference: ATP 2-01.3, para X-Y] format."
)

def extract_qa(example):
    text = example.get('text', '')
    # Gemma 2 format: <start_of_turn>user\n{system}\n\n{question}<end_of_turn>\n<start_of_turn>model\n{answer}<end_of_turn>
    u_start = text.find('<start_of_turn>user\n')
    u_end   = text.find('<end_of_turn>')
    question = ''
    if u_start != -1 and u_end != -1:
        user_block = text[u_start + len('<start_of_turn>user\n'):u_end]
        # Question follows system prompt, separated by \n\n
        parts    = user_block.split('\n\n', 1)
        question = parts[-1].strip()

    m_start  = text.find('<start_of_turn>model\n')
    m_end    = text.rfind('<end_of_turn>')
    answer   = ''
    if m_start != -1 and m_end > m_start:
        answer = text[m_start + len('<start_of_turn>model\n'):m_end].strip()

    return question, answer

def build_prompt(question):
    return (
        f"<start_of_turn>user\n{EVAL_SYSTEM}\n\n{question}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

def run_inference(model, tokenizer, question):
    prompt = build_prompt(question)
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512, temperature=0.1,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    resp = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return resp.strip()

# Load val split
val_examples = [json.loads(l) for l in open(VALID_PATH)]
questions, references, qtypes = [], [], []
for ex in val_examples:
    q, ref = extract_qa(ex)
    if q and ref:
        questions.append(q)
        references.append(ref)
        qtypes.append(ex.get('question_type', 'unknown'))
print(f'Loaded {len(questions)} validation examples')

scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=True)

# ── Base model ────────────────────────────────────────────────
print('\nLoading BASE model...')
base_model, base_tok = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(base_model)

base_preds, base_scores = [], []
for i, q in enumerate(questions):
    resp  = run_inference(base_model, base_tok, q)
    score = scorer.score(references[i], resp)['rougeL'].fmeasure
    base_preds.append(resp)
    base_scores.append(score)
    print(f'  [base] {i+1}/{len(questions)} ROUGE-L: {score:.4f}', end='\r')

del base_model, base_tok
gc.collect(); torch.cuda.empty_cache()
print(f'  [base] Done.{" "*30}')

# ── Fine-tuned model ──────────────────────────────────────────
print('\nLoading FINE-TUNED model...')
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
FastLanguageModel.for_inference(ft_model)

ft_preds, ft_scores = [], []
for i, q in enumerate(questions):
    resp  = run_inference(ft_model, ft_tok, q)
    score = scorer.score(references[i], resp)['rougeL'].fmeasure
    ft_preds.append(resp)
    ft_scores.append(score)
    print(f'  [ft]   {i+1}/{len(questions)} ROUGE-L: {score:.4f}', end='\r')

del ft_model, ft_tok
gc.collect(); torch.cuda.empty_cache()
print(f'  [ft]   Done.{" "*30}')

# ── Summary ───────────────────────────────────────────────────
avg_base = float(np.mean(base_scores))
avg_ft   = float(np.mean(ft_scores))
improved = sum(1 for b, f in zip(base_scores, ft_scores) if f > b)
delta    = avg_ft - avg_base

os.makedirs(os.path.dirname(EVAL_PATH), exist_ok=True)
with open(EVAL_PATH, 'w') as f:
    for i in range(len(questions)):
        f.write(json.dumps({
            'example':      i + 1,
            'question':     questions[i],
            'reference':    references[i],
            'base_pred':    base_preds[i],
            'ft_pred':      ft_preds[i],
            'base_rougeL':  round(base_scores[i], 6),
            'ft_rougeL':    round(ft_scores[i], 6),
            'delta':        round(ft_scores[i] - base_scores[i], 6),
            'question_type': qtypes[i],
        }) + '\n')

sign = '+' if delta >= 0 else ''
print('\n' + '='*50)
print('  EVALUATION SUMMARY (ROUGE-L)')
print('='*50)
print(f'  Examples evaluated : {len(questions)}')
print(f'  Base model ROUGE-L : {avg_base:.4f}')
print(f'  Fine-tuned ROUGE-L : {avg_ft:.4f}')
print(f'  Improvement        : {sign}{delta:.4f}  ({sign}{delta/max(avg_base,1e-9)*100:.1f}%)')
print(f'  Examples improved  : {improved} / {len(questions)}')
print('='*50)
print(f'\nResults saved → {EVAL_PATH}')

## Step 9 — Plot Results (by Question Type)

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

if 'DRIVE_BASE' not in dir():
    DRIVE_BASE = '/content/drive/MyDrive/atp-finetuning'
    EVAL_PATH  = f'{DRIVE_BASE}/eval/results.jsonl'
    CHART_PATH = f'{DRIVE_BASE}/eval/rouge_chart.png'
    VALID_PATH = f'{DRIVE_BASE}/data/val.jsonl'

results = [json.loads(l) for l in open(EVAL_PATH)]
valid   = [json.loads(l) for l in open(VALID_PATH)]

TYPE_LABELS = {
    'factual':     'Factual',
    'procedural':  'Procedural',
    'conceptual':  'Conceptual',
    'scenario':    'Scenario',
    'application': 'Application',
    'comparative': 'Comparative',
    'definition':  'Definition',
    'relationship':'Relationship',
}
TYPE_ORDER = list(TYPE_LABELS.keys())

# Aggregate by question type
buckets = defaultdict(lambda: {'base': [], 'ft': []})
for r, v in zip(results, valid):
    qt = v.get('question_type', 'unknown')
    buckets[qt]['base'].append(r['base_rougeL'])
    buckets[qt]['ft'].append(r['ft_rougeL'])

ordered   = [qt for qt in TYPE_ORDER if qt in buckets]
ordered  += [qt for qt in buckets if qt not in ordered]
labels    = [TYPE_LABELS.get(qt, qt) for qt in ordered]
base_vals = [np.mean(buckets[qt]['base']) for qt in ordered]
ft_vals   = [np.mean(buckets[qt]['ft'])   for qt in ordered]
n_vals    = [len(buckets[qt]['base'])      for qt in ordered]

x     = np.arange(len(ordered))
width = 0.35

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#0f1117')

bars_base = ax.bar(x - width/2, base_vals, width, label='Base Gemma 2-2B',
                   color='#4a90d9', alpha=0.85, zorder=3)
bars_ft   = ax.bar(x + width/2, ft_vals,   width, label='Fine-Tuned (ATP)',
                   color='#e05c5c', alpha=0.85, zorder=3)

for bar in bars_base:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8, color='#aaaaaa')
for bar in bars_ft:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.005,
            f'{h:.3f}', ha='center', va='bottom', fontsize=8, color='white', fontweight='bold')
for i, n in enumerate(n_vals):
    ax.text(x[i], -0.025, f'n={n}', ha='center', va='top', fontsize=7.5, color='#888888')

overall_base = np.mean(base_vals)
overall_ft   = np.mean(ft_vals)
pct = (overall_ft - overall_base) / overall_base * 100 if overall_base > 0 else 0

ax.set_title(
    f'ROUGE-L: Base Model vs Fine-Tuned (ATP 2-01.3 / Gemma 2-2B / Colab T4)\n'
    f'{len(results)} val examples  ·  300 QA pairs  ·  Unsloth QLoRA',
    color='white', fontsize=12, fontweight='bold', pad=14,
)
ax.set_xlabel('Question Type', color='#cccccc', fontsize=11, labelpad=24)
ax.set_ylabel('ROUGE-L Score', color='#cccccc', fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels(labels, color='#cccccc', fontsize=9)
ax.set_ylim(0, max(ft_vals + base_vals) * 1.25)
ax.tick_params(colors='#888888', which='both')
for spine in ax.spines.values():
    spine.set_edgecolor('#333333')
ax.grid(axis='y', color='#333333', linestyle='--', alpha=0.6, zorder=0)

patch_base = mpatches.Patch(color='#4a90d9', alpha=0.85,
                            label=f'Base Model  (avg {overall_base:.4f})')
patch_ft   = mpatches.Patch(color='#e05c5c', alpha=0.85,
                            label=f'Fine-tuned  (avg {overall_ft:.4f}  ·  {pct:+.1f}%)')
ax.legend(handles=[patch_base, patch_ft],
          facecolor='#1a1d27', edgecolor='#444444',
          labelcolor='#dddddd', fontsize=10, loc='upper right')

plt.tight_layout()
os.makedirs(os.path.dirname(CHART_PATH), exist_ok=True)
plt.savefig(CHART_PATH, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()
print(f'Chart saved → {CHART_PATH}')

## Step 10 — Export GGUF (Optional)

Exports the fine-tuned model to GGUF format for local deployment with Ollama.

In [ ]:
import gc, os
import torch
from unsloth import FastLanguageModel

if 'DRIVE_BASE' not in dir():
    DRIVE_BASE   = '/content/drive/MyDrive/atp-finetuning'
    ADAPTER_PATH = f'{DRIVE_BASE}/outputs/atp_gemma2_adapter'
    GGUF_DIR     = f'{DRIVE_BASE}/burns'
    MODEL_NAME   = 'unsloth/gemma-2-2b-it'

ft_model, ft_tok = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH, max_seq_length=2048, dtype=None, load_in_4bit=True,
)

os.makedirs(GGUF_DIR, exist_ok=True)
ft_model.save_pretrained_gguf(
    f'{GGUF_DIR}/atp-gemma2-colab',
    ft_tok,
    quantization_method='q4_k_m',
)
print(f'GGUF exported → {GGUF_DIR}/atp-gemma2-colab/')
print('\nTo deploy locally with Ollama:')
print('  ollama create atp-doctrine -f Modelfile')
print('  ollama run atp-doctrine')